PlantDoc 5-Fold Cross Validation Fine-Tuning

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/AgriML"

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "disease")), (
    f"Could not find src/disease under {PROJECT_ROOT}"
)
print("Using project root:", PROJECT_ROOT)

Using project root: /content/drive/MyDrive/AgriML


In [3]:
!pip install -q torch torchvision timm albumentations opencv-python pillow numpy matplotlib scikit-learn tqdm flask kaggle

In [4]:
import os

os.environ['KAGGLE_USERNAME'] = "your_kaggle_username"
os.environ['KAGGLE_KEY'] = "your_api_key_here"

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write('{"username":"%s","key":"%s"}' % (os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY']))
!chmod 600 /root/.kaggle/kaggle.json

In [5]:
KAGGLE_DL_DIR = "/content/pd_kaggle"
os.makedirs(KAGGLE_DL_DIR, exist_ok=True)

!kaggle datasets download -d abdulhasibuddin/plant-doc-dataset -p {KAGGLE_DL_DIR} --unzip

train_dir = test_dir = None
for root, dirs, _ in os.walk(KAGGLE_DL_DIR):
    if "train" in dirs and "test" in dirs:
        train_dir = os.path.join(root, "train")
        test_dir = os.path.join(root, "test")
        break

assert train_dir and test_dir, "Could not locate train and test folders after extraction"
DATA_DIR = os.path.dirname(train_dir)
print("DATA_DIR:", DATA_DIR)
print("Train classes:", sorted(os.listdir(train_dir))[:5])
print("Test classes:", sorted(os.listdir(test_dir))[:5])

Dataset URL: https://www.kaggle.com/datasets/abdulhasibuddin/plant-doc-dataset
License(s): unknown
100% 882M/882M [00:23<00:00, 39.5MB/s]

DATA_DIR: /content/pd_kaggle/PlantDoc-Dataset
Train classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot']
Test classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot']


In [6]:
import sys
import copy
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.disease.class_mapping import NUM_PD_CLASSES, PD_CLASSES, PD_CLASS_TO_IDX, PD_IDX_TO_CLASS
from src.disease.dataset import create_plantdoc_datasets, LeafDiseaseDataset, get_transforms
from src.disease.model import build_model, load_stage1_and_swap_head, freeze_backbone, unfreeze_all

In [7]:
STAGE1_PATH = os.path.join(PROJECT_ROOT, "weights", "plantvillage_pretrained.pth")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "weights")
os.makedirs(OUTPUT_DIR, exist_ok=True)

BACKBONE = "mobilenet_v3_large"
BATCH_SIZE = 16
EPOCHS = 20
WARMUP_EPOCHS = 3
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
N_SPLITS = 5
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [8]:
train_base, test_base = create_plantdoc_datasets(DATA_DIR, img_size=IMG_SIZE)
all_samples = train_base.samples + test_base.samples
labels = [s[1] for s in all_samples]
print(f"Total combined PlantDoc samples for 5-Fold CV: {len(all_samples)}")

Total combined PlantDoc samples for 5-Fold CV: 2552


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [9]:
train_transform = get_transforms(is_train=True, img_size=IMG_SIZE)
val_transform = get_transforms(is_train=False, img_size=IMG_SIZE)

class CVSubset(torch.utils.data.Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        import cv2
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        res = self.transform(image=img)
        return res["image"], label

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_results = []
all_oof_preds = []
all_oof_targets = []

In [10]:
for fold, (train_idx, val_idx) in enumerate(skf.split(all_samples, labels), 1):
    print(f"\n--- Running Fold {fold}/{N_SPLITS} ---")

    train_fold_samples = [all_samples[i] for i in train_idx]
    val_fold_samples = [all_samples[i] for i in val_idx]

    train_ds = CVSubset(train_fold_samples, train_transform)
    val_ds = CVSubset(val_fold_samples, val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    if os.path.exists(STAGE1_PATH):
        model = load_stage1_and_swap_head(STAGE1_PATH, new_num_classes=NUM_PD_CLASSES, backbone=BACKBONE)
    else:
        model = build_model(num_classes=NUM_PD_CLASSES, backbone=BACKBONE, pretrained_imagenet=True)
    model = model.to(DEVICE)

    freeze_backbone(model, BACKBONE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0
    best_model_state = None

    for epoch in range(1, EPOCHS + 1):
        if epoch == WARMUP_EPOCHS + 1:
            unfreeze_all(model)
            if BACKBONE == "resnet34":
                backbone_p = [p for n, p in model.named_parameters() if not n.startswith("fc")]
                head_p = model.fc.parameters()
            else:
                backbone_p = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
                head_p = model.classifier.parameters()
            optimizer = torch.optim.AdamW([
                {"params": backbone_p, "lr": LR_BACKBONE},
                {"params": head_p, "lr": LR_HEAD}
            ], weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(EPOCHS - WARMUP_EPOCHS))

        model.train()
        for imgs, tgts in train_loader:
            imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, tgts)
            loss.backward()
            optimizer.step()

        model.eval()
        v_corr, v_tot = 0, 0
        preds_list, tgts_list = [], []
        with torch.no_grad():
            for imgs, tgts in val_loader:
                imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
                out = model(imgs)
                _, p = torch.max(out, 1)
                v_corr += (p == tgts).sum().item()
                v_tot += tgts.size(0)
                preds_list.extend(p.cpu().tolist())
                tgts_list.extend(tgts.cpu().tolist())

        val_acc = (v_corr / v_tot) * 100.0
        scheduler.step()

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            best_preds = preds_list
            best_tgts = tgts_list

    print(f"Fold {fold} Best Validation Accuracy: {best_acc:.2f}%")
    fold_results.append(best_acc)
    all_oof_preds.extend(best_preds)
    all_oof_targets.extend(best_tgts)

    fold_path = os.path.join(OUTPUT_DIR, f"plantdoc_fold{fold}.pth")
    torch.save({
        "fold": fold,
        "val_acc": best_acc,
        "num_classes": NUM_PD_CLASSES,
        "backbone": BACKBONE,
        "model_state_dict": best_model_state,
        "class_to_idx": PD_CLASS_TO_IDX,
        "idx_to_class": PD_IDX_TO_CLASS
    }, fold_path)

mean_acc = np.mean(fold_results)
std_acc = np.std(fold_results)
print(f"\n5-Fold CV Mean Accuracy: {mean_acc:.2f}% (+/- {std_acc:.2f}%)")


--- Running Fold 1/5 ---
Fold 1 Best Validation Accuracy: 61.84%

--- Running Fold 2/5 ---
Fold 2 Best Validation Accuracy: 62.04%

--- Running Fold 3/5 ---
Fold 3 Best Validation Accuracy: 60.20%

--- Running Fold 4/5 ---
Fold 4 Best Validation Accuracy: 57.45%

--- Running Fold 5/5 ---
Fold 5 Best Validation Accuracy: 60.20%

5-Fold CV Mean Accuracy: 60.34% (+/- 1.64%)


In [11]:
best_fold_idx = int(np.argmax(fold_results)) + 1
best_fold_ckpt = os.path.join(OUTPUT_DIR, f"plantdoc_fold{best_fold_idx}.pth")
final_path = os.path.join(OUTPUT_DIR, "leaf_disease_model_final.pth")

import shutil
shutil.copy2(best_fold_ckpt, final_path)
print(f"Saved Fold {best_fold_idx} checkpoint as final model: {final_path}")

target_names = [PD_IDX_TO_CLASS[i] for i in range(NUM_PD_CLASSES)]
print("\n=== Out-Of-Fold Classification Report ===")
print(classification_report(all_oof_targets, all_oof_preds, target_names=target_names, zero_division=0))

Saved Fold 2 checkpoint as final model: /content/drive/MyDrive/AgriML/weights/leaf_disease_model_final.pth

=== Out-Of-Fold Classification Report ===
                               precision    recall  f1-score   support

             Apple Cedar Rust       0.69      0.69      0.69        89
                Apple Healthy       0.74      0.75      0.74        91
                   Apple Scab       0.77      0.57      0.66        87
   Bell Pepper Bacterial Spot       0.50      0.44      0.47        71
          Bell Pepper Healthy       0.56      0.57      0.57        61
            Blueberry Healthy       0.63      0.69      0.66       116
               Cherry Healthy       0.69      0.65      0.67        57
             Corn Common Rust       0.75      0.79      0.77       116
          Corn Gray Leaf Spot       0.53      0.41      0.46        68
    Corn Northern Leaf Blight       0.70      0.83      0.76       192
              Grape Black Rot       0.75      0.69      0.72        